# Milestone 5 - running someone else's code without it taking you down

**Why this matters.** This is the hardest milestone and the most transferable.
Any system that executes work it did not write - a CI runner, a plugin host, a
notebook kernel like the one running this page - faces exactly this: run it
elsewhere, bound how long it gets, and turn every possible failure into data
instead of an exception.

You do not have to write the child process. `drill._harness` already does that
job. Your task is the **parent** side: launch it, feed it, enforce the timeout,
and parse what comes back.

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")

import json
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any

import drill._harness
# The child prints this line, then one line of JSON. Anything printed before it is
# the submission's own output and must be ignored.
from drill._harness import SENTINEL
from drill.problems import get_problem

HARNESS = Path(drill._harness.__file__)
TWO_SUM = get_problem("two_sum")
CORRECT = (
    "def two_sum(nums, target):\n"
    "    seen = {}\n"
    "    for i, n in enumerate(nums):\n"
    "        if target - n in seen:\n"
    "            return [seen[target - n], i]\n"
    "        seen[n] = i\n"
    "    return []\n"
)

Before writing anything, look at what you are working with: a real problem from
the bank, and the harness's protocol.

In [ ]:
print(TWO_SUM.function_name, "unordered:", TWO_SUM.unordered)
TWO_SUM.test_cases[:2]

## 1. `build_payload`

Build the JSON payload the harness expects. It needs exactly these keys:

```python
{
  "source":        the submitted code, as a string,
  "function_name": which function to call,
  "unordered":     bool - compare ignoring order,
  "cases":         [{"args": [...], "expected": ...}, ...],
}
```

Note `args` must be a **list**, not a tuple: `json.dumps` cannot tell you apart
from a tuple, it just writes an array, and the child reads it back as a list.
Getting this wrong is the kind of serialisation mismatch that produces a baffling
error three layers away.

In [ ]:
def build_payload(problem: Any, source: str) -> dict:
    """Build the JSON payload the harness expects."""
    # TODO: implement.
    raise NotImplementedError("Milestone 5: implement build_payload")

In [ ]:
%%ipytest -qq

def test_payload_has_the_required_keys():
    payload = build_payload(TWO_SUM, CORRECT)
    assert set(payload) >= {"source", "function_name", "unordered", "cases"}
    assert payload["function_name"] == "two_sum"


def test_payload_args_are_lists_not_tuples():
    """JSON has no tuples. Send lists, or the child sees something you did not mean."""
    payload = build_payload(TWO_SUM, CORRECT)
    assert all(isinstance(case["args"], list) for case in payload["cases"])


def test_payload_is_json_serialisable():
    json.dumps(build_payload(TWO_SUM, CORRECT))

## 2. `parse_report`

Pull the JSON report out of the child's stdout. The submission is allowed to
print - you **will** print things while debugging - so the report is whatever
follows the *last* `SENTINEL` line. Return the decoded dict, or `None` if there is
no sentinel or the JSON will not parse.

In [ ]:
def parse_report(stdout: str) -> dict | None:
    """Pull the JSON report out of the child's stdout."""
    # TODO: implement.
    raise NotImplementedError("Milestone 5: implement parse_report")

In [ ]:
%%ipytest -qq

def test_parse_report_reads_json_after_the_sentinel():
    assert parse_report(f'{SENTINEL}\n{{"cases": [], "fatal_error": null}}')["cases"] == []


def test_parse_report_ignores_anything_printed_first():
    stdout = f'debugging output\nmore noise\n{SENTINEL}\n{{"cases": [], "fatal_error": null}}'
    assert parse_report(stdout) is not None


def test_parse_report_returns_none_without_a_sentinel():
    assert parse_report("just some output") is None


def test_parse_report_returns_none_on_bad_json():
    assert parse_report(f"{SENTINEL}\nnot json at all") is None

## 3. `run_submission`

Run `source` against `problem`'s test cases and report what happened. Return a
dict with these keys:

| Key | Type | Meaning |
|---|---|---|
| `passed` | bool | every case passed (and there was at least one) |
| `total` | int | |
| `passed_count` | int | |
| `cases` | list | per-case dicts from the harness |
| `fatal_error` | str or None | |
| `timed_out` | bool | |

This function must **never** raise because of a bad submission. A syntax error, a
missing function, a crash, an infinite loop - each is a normal outcome that comes
back as a result with `passed=False`. If any of them escaped as an exception, one
bad submission would end the practice session.

How to run the child (write the payload to a temp file first):

```python
subprocess.run(
    [sys.executable, "-I", str(HARNESS), <payload path>],
    capture_output=True, text=True, timeout=timeout,
)
```

`-I` is isolated mode: the child ignores `PYTHONPATH` and the user site
directory, so the submission cannot import your project and shadow something.
Catch `subprocess.TimeoutExpired` and return `timed_out=True`.

In [ ]:
def run_submission(problem: Any, source: str, timeout: int = 10) -> dict:
    """Run `source` against `problem`'s test cases and report what happened."""
    # TODO: implement.
    raise NotImplementedError("Milestone 5: implement run_submission")

In [ ]:
%%ipytest -qq

def test_correct_solution_passes():
    result = run_submission(TWO_SUM, CORRECT)
    assert result["passed"]
    assert result["passed_count"] == result["total"] == len(TWO_SUM.test_cases)
    assert result["fatal_error"] is None


def test_wrong_answer_fails_without_raising():
    result = run_submission(TWO_SUM, "def two_sum(nums, target):\n    return [0, 1]\n")
    assert not result["passed"]
    assert 0 < result["passed_count"] < result["total"]


def test_syntax_error_comes_back_as_a_result():
    result = run_submission(TWO_SUM, "def two_sum(nums, target)\n    return []\n")
    assert not result["passed"]
    assert result["fatal_error"]


def test_missing_function_comes_back_as_a_result():
    result = run_submission(TWO_SUM, "def nope():\n    return []\n")
    assert not result["passed"]
    assert result["fatal_error"]


def test_infinite_loop_times_out_instead_of_hanging():
    result = run_submission(
        TWO_SUM, "def two_sum(nums, target):\n    while True:\n        pass\n", timeout=2
    )
    assert result["timed_out"]
    assert not result["passed"]


def test_a_printing_submission_still_grades():
    assert run_submission(TWO_SUM, 'print("hello")\n' + CORRECT)["passed"]

## Try it

Write your own submission here - a wrong one is more interesting - and look at
what comes back.

In [ ]:
mine = """
def two_sum(nums, target):
    for i in range(len(nums)):
        for j in range(len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
"""
run_submission(TWO_SUM, mine)